# Pathogen Atlas V3 workflow

Run Phases 1–5 in order. Phase 1 builds the complete master taxonomy. Phases 2–5 use the selected organism groups in `configs/default.yaml`. This notebook reviews those settings, source modules, and local inputs before running phases.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml

ROOT = Path.cwd()
CONFIG_PATH = ROOT / "configs/default.yaml"
CONFIG = load_yaml(CONFIG_PATH)
SELECTED_GROUPS = selected_organism_groups(CONFIG)
print("Project:", CONFIG["project"]["id"])
print("Supported organism groups:", ", ".join(CONFIG["project"]["domains"]))
print("Groups selected for Phases 2–5:", ", ".join(SELECTED_GROUPS))
print("To change the run, edit run.organism_groups in", CONFIG_PATH)
print("Phase 1 taxonomy config:", ROOT / "configs/taxonomy/source_files.yaml")
print("PubMed output root:", CONFIG["paths"].get("pubmed_domain_root", "<legacy path layout>"))
print("Decision ledger:", CONFIG["paths"]["decision_ledger"])
print("Release output:", CONFIG["paths"]["release_root"])
print("Pathogenic Interest Index version: pathogenic-interest-index-v3.1")
display(pd.DataFrame([{"phase": 1, "scope": "complete taxonomy", "module": "pathogen_atlas_v3.pipelines.phase1_taxonomy"}, {"phase": 2, "scope": ", ".join(SELECTED_GROUPS), "module": "pathogen_atlas_v3.pipelines.phase2_pubmed"}, {"phase": 3, "scope": ", ".join(SELECTED_GROUPS), "module": "pathogen_atlas_v3.pipelines.phase3_decisions"}, {"phase": 4, "scope": "saved decisions", "module": "pathogen_atlas_v3.pipelines.phase4_release"}, {"phase": 5, "scope": ", ".join(SELECTED_GROUPS), "module": "pathogen_atlas_v3.pipelines.phase5_embedding"}]))

## How to select domains

Edit `run.organism_groups` in `configs/default.yaml` to one or more of `bacteria`, `fungi`, and `viruses`. Phase 1 keeps the master taxonomy complete; the selected list controls retrieval, scoring, and embedding work. Each Phase 2 query and article cache is stored under a domain-specific directory.

All network work is resumable. Phase 2 exposes query/fetch limits and ETA logs. Phase 3 has a no-network preflight, bounded calls, concurrency, checkpoint intervals, and OpenRouter balance reporting. Phase 4 only rebuilds local tables. Phase 5 exposes the IMSTE settings.

## Main implementation files

- Phase 1: `pathogen_atlas_v3/pipelines/phase1_taxonomy.py`, `pathogen_atlas_v3/taxonomy/adapters.py`
- Phase 2: `pathogen_atlas_v3/pipelines/phase2_pubmed.py`, `pathogen_atlas_v3/abstracts/pubmed.py`
- Phase 3: `pathogen_atlas_v3/pipelines/phase3_decisions.py`, `pathogen_atlas_v3/decisions/runner.py`, `pathogen_atlas_v3/decisions/jev.py`
- Phase 4: `pathogen_atlas_v3/pipelines/phase4_release.py`
- Phase 5: `pathogen_atlas_v3/pipelines/phase5_embedding.py`, `pathogen_atlas_v3/embeddings/imste.py`

The notebooks call these named source functions directly. They do not hide a multi-phase run behind one generic orchestration function.